<!-- nav -->
[← 1 · Rules as data](01_Rules_As_Data.ipynb) · [Home](../../README.md) · [3 · Rules over collections →](03_Rules_Over_Collections.ipynb)

# 2 · Answers you can trust

**The problem.** Loggers drop out: a battery dies, a radio loses signal, and a shipment arrives with no temperature on
file. What should "is it in range?" answer then? A Python function raises `TypeError` comparing `None` with `2.0`. A
JavaScript one compares `undefined` and says `false`: the shipment is quietly rejected, or, with the test written the
other way round, quietly accepted. Neither is an answer you can act on.

An expression answers **unknown** when the data doesn't decide, and says so. This case study shows how unknowns
combine, how you decide what they mean, and why `1` and `1.0` aren't equal.

In [1]:
from mbse.Expressions import Evaluators, Expressions as E
from mbse.Expressions.Dialects.Python import Text
from toolkit import cold_chain, show

store = cold_chain()
s1 = store.Shipment().id("S1").celsius(5.5).hours(30).create()
s2 = store.Shipment().id("S2").celsius(9.1).hours(80).create()
s3 = store.Shipment().id("S3").hours(20).create()  # its logger never reported
shipments = [s1, s2, s3]
in_range = Text.FromFunction(lambda this: 2.0 <= this.celsius <= 8.0)

## Step 1: absent is unknown

In mbse-schemas, a property is either present or absent; there's no `None` value to compare. Reading an absent
property gives **unknown**, which Python writes `None`, and a comparison with unknown is unknown:

In [2]:
{shipment.id: Evaluators.predicate(in_range, shipment) for shipment in shipments}

{'S1': True, 'S2': False, 'S3': None}

S3's answer isn't false: it's that nobody knows. `has` asks whether a property is present, which always has an
answer. In a lambda, `is not None` is `has`:

In [3]:
reported = Text.FromFunction(lambda this: this.celsius is not None)
print(show(reported))
{shipment.id: Evaluators.predicate(reported, shipment) for shipment in shipments}

has(this, 'celsius')


{'S1': True, 'S2': True, 'S3': False}

## Step 2: how unknowns combine

`and`, `or`, `not` and `implies` follow **Kleene's three-valued logic**: an unknown operand doesn't matter when the
other one decides. `False and unknown` is false, whatever the unknown turns out to be; `True or unknown` is true.

In [4]:
late_or_out = Text.FromFunction(lambda this: this.hours > 72 or not (2.0 <= this.celsius <= 8.0))
ok_or_silent = Text.FromFunction(lambda this: this.celsius is None or 2.0 <= this.celsius <= 8.0)
for rule in (late_or_out, ok_or_silent):
    print(show(rule))
    print({shipment.id: Evaluators.predicate(rule, shipment) for shipment in shipments})

or(gt(get(this, 'hours'), 72), not(and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0))))
{'S1': False, 'S2': True, 'S3': None}
or(not(has(this, 'celsius')), and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0)))
{'S1': True, 'S2': False, 'S3': True}


For S3, the first rule is unknown: it was on time, so everything rests on a temperature nobody has. The second
decides S3 at its first operand, and, as in Python, doesn't evaluate the second.

`implies` reads "if ... then": a rule about only some shipments holds trivially for the others:

In [5]:
long_trips_cold = E.variable("this").hours.gt(72).implies(E.variable("this").celsius.le(8.0))
print(show(long_trips_cold))
{shipment.id: Evaluators.predicate(long_trips_cold, shipment) for shipment in shipments}

implies(gt(get(this, 'hours'), 72), le(get(this, 'celsius'), 8.0))


{'S1': True, 'S2': False, 'S3': True}

Python has no operator for implication, so this rule is written with writers.

## Step 3: you decide what unknown means

An evaluator never guesses. The program asking decides, once, at the point where it acts. A dashboard that flags
shipments to inspect flags what isn't known to be fine; an audit that certifies shipments certifies only what is known
to be fine:

In [6]:
inspect = [shipment.id for shipment in shipments if Evaluators.predicate(in_range, shipment) is not True]
certify = [shipment.id for shipment in shipments if Evaluators.predicate(in_range, shipment) is True]
inspect, certify

(['S2', 'S3'], ['S1'])

Writing `if Evaluators.predicate(...)` would treat unknown as false, which is the silent mistake this case study
started with. Compare with `is True` or `is False`.

## Step 4: no coercion

An `int` and a `float` are different kinds of value, as they are in mbse-schemas: `1 == 1.0` isn't true, it's
unknown, because the two aren't comparable. That catches a classic slip. `celsius` is a float, so a rule that writes
the limit as an int answers unknown for every shipment, rather than an answer that depends on the language:

In [7]:
sloppy = Text.FromFunction(lambda this: 2 <= this.celsius <= 8)
{shipment.id: Evaluators.predicate(sloppy, shipment) for shipment in shipments}

{'S1': None, 'S2': None, 'S3': None}

Python would have compared `2 <= 5.5` happily, and TypeScript compares an integer `2n` with `5.5` too, but by its own
rules. A rule meant to give the same answer everywhere can't depend on either. Arithmetic is stricter still: adding an
int and a float is an error, raised when the rule runs:

In [8]:
try:
    Evaluators.OfAny(Text.FromFunction(lambda this: this.hours + 0.5 > 72.0), {"this": s1})
except TypeError as error:
    print(error)

add expects numbers of one domain, got int and float


## Step 5: what raises

Unknown is for data that doesn't decide. A rule that can't be evaluated at all raises: a variable nothing binds, an
operation outside the core vocabulary, an operand of the wrong kind. Those are mistakes in the rule, not in the data,
and `validate` (case study 1) reports the first two before the rule runs.

In [9]:
for rule, scope in [(in_range, {}), (E.operation("div", 1, 2), {}), (E.variable("this").id.add(1), {"this": s1})]:
    try:
        Evaluators.OfAny(rule, scope)
    except Exception as error:
        print(type(error).__name__, error)

KeyError "variable 'this' is not bound"
NotImplementedError 'div' is not a core operation
TypeError add expects numbers of one domain, got str and int


## Why it works this way

- **Unknown is not false.** Treating it as false fails good records silently, and treating it as true passes bad ones.
  Three values keep the question open until a program, knowing what it's for, closes it.
- **Kleene, because it's the logic of not knowing yet.** `False and unknown` is false because no value of the unknown
  could make it true. Every answer an expression gives is one that more data couldn't overturn.
- **No coercion, because languages disagree.** Python, TypeScript, MATLAB, Excel and C each convert numbers by their
  own rules. A rule that relied on any of them would mean different things in different places.

<!-- nav -->
[← 1 · Rules as data](01_Rules_As_Data.ipynb) · [Home](../../README.md) · [3 · Rules over collections →](03_Rules_Over_Collections.ipynb)